<a href="https://colab.research.google.com/github/devsomya28/ML_PRACTICE/blob/main/movie_text_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd
from tqdm import tqdm

In [2]:
import requests

genre_url = "https://api.themoviedb.org/3/genre/movie/list?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US"

response = requests.get(genre_url)

genre_data = response.json()

genre_data

{'genres': [{'id': 28, 'name': 'Action'},
  {'id': 12, 'name': 'Adventure'},
  {'id': 16, 'name': 'Animation'},
  {'id': 35, 'name': 'Comedy'},
  {'id': 80, 'name': 'Crime'},
  {'id': 99, 'name': 'Documentary'},
  {'id': 18, 'name': 'Drama'},
  {'id': 10751, 'name': 'Family'},
  {'id': 14, 'name': 'Fantasy'},
  {'id': 36, 'name': 'History'},
  {'id': 27, 'name': 'Horror'},
  {'id': 10402, 'name': 'Music'},
  {'id': 9648, 'name': 'Mystery'},
  {'id': 10749, 'name': 'Romance'},
  {'id': 878, 'name': 'Science Fiction'},
  {'id': 10770, 'name': 'TV Movie'},
  {'id': 53, 'name': 'Thriller'},
  {'id': 10752, 'name': 'War'},
  {'id': 37, 'name': 'Western'}]}

In [3]:
genre_dict = {}

for genre in genre_data["genres"]:
    genre_dict[genre["id"]] = genre["name"]

genre_dict

{28: 'Action',
 12: 'Adventure',
 16: 'Animation',
 35: 'Comedy',
 80: 'Crime',
 99: 'Documentary',
 18: 'Drama',
 10751: 'Family',
 14: 'Fantasy',
 36: 'History',
 27: 'Horror',
 10402: 'Music',
 9648: 'Mystery',
 10749: 'Romance',
 878: 'Science Fiction',
 10770: 'TV Movie',
 53: 'Thriller',
 10752: 'War',
 37: 'Western'}

In [4]:
movies = []

for page in tqdm(range(1, 6)):      # Only 5 pages for testing

    url = f"https://api.themoviedb.org/3/movie/top_rated?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US&page={page}"

    response = requests.get(url)

    data = response.json()

    for movie in data["results"]:

        genres = []

        for gid in movie["genre_ids"]:
            genres.append(genre_dict.get(gid))

        movies.append({
            "Movie Name": movie["title"],
            "Description": movie["overview"],
            "Genre": ", ".join(genres)
        })

100%|██████████| 5/5 [00:02<00:00,  2.23it/s]


In [5]:
df = pd.DataFrame(movies)

df.head()

,Movie Name,Description,Genre
0,Avatar Aang: The Last Airbender,"Avatar Aang, the world's last Airbender, learn...","Animation, Action, Adventure, Fantasy"
1,Accidental Partners,Two women discover they were both scammed by t...,"Comedy, Romance"
2,Swapped,"A small woodland creature and a majestic bird,...","Adventure, Animation, Family, Fantasy"
3,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,"Drama, Crime"
4,Michael,"The story of Michael Jackson, one of the most ...","Music, Drama"


In [6]:
df.to_csv("movies.csv", index=False)

print("Dataset Saved Successfully!")

Dataset Saved Successfully!


In [7]:
df["Clean_Description"] = df["Description"]

df[["Description", "Clean_Description"]].head()

,Description,Clean_Description
0,"Avatar Aang, the world's last Airbender, learn...","Avatar Aang, the world's last Airbender, learn..."
1,Two women discover they were both scammed by t...,Two women discover they were both scammed by t...
2,"A small woodland creature and a majestic bird,...","A small woodland creature and a majestic bird,..."
3,Imprisoned in the 1940s for the double murder ...,Imprisoned in the 1940s for the double murder ...
4,"The story of Michael Jackson, one of the most ...","The story of Michael Jackson, one of the most ..."


In [8]:
df["Clean_Description"] = df["Clean_Description"].str.lower()

df["Clean_Description"].head()

,Clean_Description
0,"avatar aang, the world's last airbender, learn..."
1,two women discover they were both scammed by t...
2,"a small woodland creature and a majestic bird,..."
3,imprisoned in the 1940s for the double murder ...
4,"the story of michael jackson, one of the most ..."


In [9]:
from bs4 import BeautifulSoup

def remove_html(text):
    return BeautifulSoup(str(text), "html.parser").get_text()

df["Clean_Description"] = df["Clean_Description"].apply(remove_html)

In [10]:
import re

def remove_url(text):
    return re.sub(r'https?://\S+|www\.\S+', '', str(text))

df["Clean_Description"] = df["Clean_Description"].apply(remove_url)

In [11]:
import string

def remove_punc(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df["Clean_Description"] = df["Clean_Description"].apply(remove_punc)

In [12]:
!pip install textblob

In [13]:
from textblob import TextBlob

def correct_spelling(text):
    return str(TextBlob(text).correct())

df.loc[:9, "Clean_Description"] = df.loc[:9, "Clean_Description"].apply(correct_spelling)

In [14]:
import nltk

nltk.download('stopwords')

from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

df["Clean_Description"] = df["Clean_Description"].apply(remove_stopwords)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [15]:
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.tokenize import word_tokenize

df["Tokens"] = df["Clean_Description"].apply(word_tokenize)

df["Tokens"].head()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


,Tokens
0,"[altar, sang, worlds, last, airbender, learns,..."
1,"[two, women, discover, slammed, man, also, got..."
2,"[small, woodland, creature, majestic, bird, tw..."
3,"[imprisoned, 1940s, double, murder, wife, love..."
4,"[story, michael, jackson, one, influential, ar..."


In [16]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

def stem_words(tokens):
    return [stemmer.stem(word) for word in tokens]

df["Stemmed"] = df["Tokens"].apply(stem_words)

In [17]:
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_words(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

df["Lemmatized"] = df["Tokens"].apply(lemmatize_words)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [18]:
df[[
    "Movie Name",
    "Description",
    "Clean_Description",
    "Tokens",
    "Stemmed",
    "Lemmatized",
    "Genre"
]].head()

,Movie Name,Description,Clean_Description,Tokens,Stemmed,Lemmatized,Genre
0,Avatar Aang: The Last Airbender,"Avatar Aang, the world's last Airbender, learn...",altar sang worlds last airbender learns ancien...,"[altar, sang, worlds, last, airbender, learns,...","[altar, sang, world, last, airbend, learn, anc...","[altar, sang, world, last, airbender, learns, ...","Animation, Action, Adventure, Fantasy"
1,Accidental Partners,Two women discover they were both scammed by t...,two women discover slammed man also got pregna...,"[two, women, discover, slammed, man, also, got...","[two, women, discov, slam, man, also, got, pre...","[two, woman, discover, slammed, man, also, got...","Comedy, Romance"
2,Swapped,"A small woodland creature and a majestic bird,...",small woodland creature majestic bird two natu...,"[small, woodland, creature, majestic, bird, tw...","[small, woodland, creatur, majest, bird, two, ...","[small, woodland, creature, majestic, bird, tw...","Adventure, Animation, Family, Fantasy"
3,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,imprisoned 1940s double murder wife lover stan...,"[imprisoned, 1940s, double, murder, wife, love...","[imprison, 1940, doubl, murder, wife, lover, s...","[imprisoned, 1940s, double, murder, wife, love...","Drama, Crime"
4,Michael,"The story of Michael Jackson, one of the most ...",story michael jackson one influential artists ...,"[story, michael, jackson, one, influential, ar...","[stori, michael, jackson, one, influenti, arti...","[story, michael, jackson, one, influential, ar...","Music, Drama"


In [19]:
df.to_csv("movies_preprocessed.csv", index=False)

print("✅ Assignment Completed Successfully!")

✅ Assignment Completed Successfully!
